# 8.3. Network in Network \(NiN\)

Last chapter we saw how VGG introduced the revolutionary concept at its time of thinking of neural networks in terms of _blocks_ instead of individual layers. However, an issue with VGG, AlexNet, LeNet and the likes is the FC layers at the head of the network which require a very large amount of parameters, translating to inefficient device memory usage.

Shortly after VGG was published, an improved architecture called _network in network_ \(NiN\) was introduced. Its key innovations are listed below.

1. Using $1 \times 1$ convolutions as FC layers acting on the channel dimension and operating on each individual pixel
1. Replacing the final FC layers from VGG / AlexNet at the head of the neural network with a _global average pooling_ layer followed by a flattening layer at the head of the NiN family of networks. This enabled the total number of model parameters to be dramatically reduced thus improving the utilization of device memory, albeit at the cost of additional training time

This notebook is based on the cloud-based [AtomGit AI Notebook Lab](https://ai.gitcode.com/docs/notebooks/free-usage/) environment with datacenter-level Ascend 910B4 training-optimized NPUs. The software versions used by this notebook are listed below.

1. Ubuntu 22.04 LTS
1. Python 3.11
1. MindSpore 2.8.0
1. CANN 8.5.0

In [1]:
!npu-smi info

+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.1                   Version: 25.5.1                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                      | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 4     910B4               | OK            | 96.7        40                0    / 0             |
| 0                         | 0000:81:00.0  | 0           0    / 0          2870 / 32768         |
+===========================+===============+====================================================+
+---------------------------+---------------+----------------------------------------------------+
| NPU     

In [2]:
%pip install mindspore==2.8.0 \
    -i https://repo.mindspore.cn/pypi/simple \
    --trusted-host repo.mindspore.cn \
    --extra-index-url https://repo.huaweicloud.com/repository/pypi/simple

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://repo.mindspore.cn/pypi/simple, https://repo.huaweicloud.com/repository/pypi/simple

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/python3.11.14/lib/python3.11/site-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


MindSpore version:  2.8.0


[WARNING] DEVICE(15206,ffff03f7f120,python3.11):2026-05-11-22:37:47.813.212 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:176] CheckVmmDriverVersion] Open file /etc/ascend_install.info failed.
[WARNING] DEVICE(15206,ffff03f7f120,python3.11):2026-05-11-22:37:47.813.262 [mindspore/ccsrc/plugin/ascend/res_manager/mem_manager/ascend_vmm_adapter.h:204] CheckVmmDriverVersion] Open file /usr/local/Ascend/driver/version.info failed.


The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 8.3.1. NiN Blocks

Each NiN block consists of the following layers.

1. 1 convolution layer with customizable kernel size, stride and padding followed by ReLU activation
1. 2 $1 \times 1$ convolutions acting as per-pixel FC layers operating on the channel dimension, each followed by ReLU activation

Apart from the hyperparameters of the 1st convolution layer, the number of \(output\) channels is also a hyperparameter to each NiN block applied equally to all 3 convolution layers.

The original NiN block implementation did not include batch normalization but we include it here after each ReLU activation to stabilize training under MindSpore's `O2` mixed precision mode which performs most operations in half precision \(FP16\).

In [4]:
import mindspore.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride=1, pad_mode='same', padding=0):
    return nn.SequentialCell([
        nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, pad_mode=pad_mode, padding=padding),
        nn.ReLU(),
        nn.BatchNorm2d(out_channels),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.BatchNorm2d(out_channels),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(),
        nn.BatchNorm2d(out_channels)
    ])

nin_3x3 = nin_block(3, 64, kernel_size=3)
nin_3x3

SequentialCell(
  (0): Conv2d(input_channels=3, output_channels=64, kernel_size=(3, 3), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff0c07c390>, bias_init=None, format=NCHW)
  (1): ReLU()
  (2): BatchNorm2d(num_features=64, eps=1e-05, momentum=0.9, gamma=Parameter (name=2.gamma, shape=(64,), dtype=Float32, requires_grad=True), beta=Parameter (name=2.beta, shape=(64,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=2.moving_mean, shape=(64,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=2.moving_variance, shape=(64,), dtype=Float32, requires_grad=False))
  (3): Conv2d(input_channels=64, output_channels=64, kernel_size=(1, 1), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff0c07c610>, bias_init=None, format=NCHW)
  (4): ReLU()
  (5): BatchNorm

## 8.3.2 NiN Model

NiN uses the same initial convolution sizes block-wise as AlexNet: $11 \times 11$, followed by $5 \times 5$ and $3 \times 3$. It also uses the same number of output channels block-wise. Each NiN block is followed by a max pooling layer with a $3 \times 3$ window and stride 2.

Another major difference of NiN over VGG / AlexNet is the absence of the final FC layers near the head of the model. Instead, we have a global average pooling layer yielding a $1 \times 1$ feature map with the each output channel corresponding to a class label, followed by a final flattening layer.

In [5]:
nin_full = nn.SequentialCell([
    nin_block(1, 96, kernel_size=11, stride=4, pad_mode='pad', padding=0),
    nn.MaxPool2d(3, stride=2),
    nin_block(96, 256, kernel_size=5),
    nn.MaxPool2d(3, stride=2),
    nin_block(256, 384, kernel_size=3),
    nn.MaxPool2d(3, stride=2),
    nn.Dropout(p=0.5),
    nin_block(384, 10, kernel_size=3),
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten()
])
nin_full

SequentialCell(
  (0): SequentialCell(
    (0): Conv2d(input_channels=1, output_channels=96, kernel_size=(11, 11), stride=(4, 4), pad_mode=pad, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff0c498350>, bias_init=None, format=NCHW)
    (1): ReLU()
    (2): BatchNorm2d(num_features=96, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.2.gamma, shape=(96,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.2.beta, shape=(96,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.2.moving_mean, shape=(96,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.2.moving_variance, shape=(96,), dtype=Float32, requires_grad=False))
    (3): Conv2d(input_channels=96, output_channels=96, kernel_size=(1, 1), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff0c07b510>, bias_init=None, for

Let's inspect the output shape of each block.

In [6]:
import mindspore.amp as amp

nin_full_amp = amp.auto_mixed_precision(network=nin_full, amp_level='O2')
nin_full_amp

_OutputTo32(
  (_backbone): SequentialCell(
    (0): SequentialCell(
      (0): Conv2d(input_channels=1, output_channels=96, kernel_size=(11, 11), stride=(4, 4), pad_mode=pad, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<mindspore.common.initializer.HeUniform object at 0xffff0c498350>, bias_init=None, format=NCHW)
      (1): ReLU()
      (2): _OutputTo16(
        (_backbone): BatchNorm2d(num_features=96, eps=1e-05, momentum=0.9, gamma=Parameter (name=0.2.gamma, shape=(96,), dtype=Float32, requires_grad=True), beta=Parameter (name=0.2.beta, shape=(96,), dtype=Float32, requires_grad=True), moving_mean=Parameter (name=0.2.moving_mean, shape=(96,), dtype=Float32, requires_grad=False), moving_variance=Parameter (name=0.2.moving_variance, shape=(96,), dtype=Float32, requires_grad=False))
      )
      (3): Conv2d(input_channels=96, output_channels=96, kernel_size=(1, 1), stride=(1, 1), pad_mode=same, padding=0, dilation=(1, 1), group=1, has_bias=False, weight_init=<minds

In [7]:
import mindspore.ops as ops

def layer_summary(net, X_shape):
    print(f'Input shape: {X_shape}')
    X = ops.randn(*X_shape)
    for cell in net.cells():
        X = cell(X)
        print(f'Output shape from {cell.__class__.__name__}: {X.shape}')

X_shape = (1, 1, 224, 224)
layer_summary(net=nin_full, X_shape=X_shape)

Input shape: (1, 1, 224, 224)
Output shape from SequentialCell: (1, 96, 54, 54)
Output shape from MaxPool2d: (1, 96, 26, 26)
Output shape from SequentialCell: (1, 256, 26, 26)
Output shape from MaxPool2d: (1, 256, 12, 12)
Output shape from SequentialCell: (1, 384, 12, 12)
Output shape from MaxPool2d: (1, 384, 5, 5)
Output shape from Dropout: (1, 384, 5, 5)
Output shape from SequentialCell: (1, 10, 5, 5)
Output shape from AdaptiveAvgPool2d: (1, 10, 1, 1)
Output shape from Flatten: (1, 10)


## 8.3.3. Training

Let's train NiN on the Fashion MNIST dataset and compare its validation loss and accuracy with previous examples such as AlexNet and VGG-11.

In [8]:
import os

dataset_dir = 'data/fashion/'
os.makedirs(dataset_dir, exist_ok=True)

In [9]:
import gzip
import urllib.request

prefix_url = 'https://donaldsebleung.com/assets/datasets/fashion-mnist'
X_train_url = f'{prefix_url}/train-images-idx3-ubyte.gz'
y_train_url = f'{prefix_url}/train-labels-idx1-ubyte.gz'
X_test_url = f'{prefix_url}/t10k-images-idx3-ubyte.gz'
y_test_url = f'{prefix_url}/t10k-labels-idx1-ubyte.gz'

X_train_path = os.path.join(dataset_dir, 'train-images-idx3-ubyte')
y_train_path = os.path.join(dataset_dir, 'train-labels-idx1-ubyte')
X_test_path = os.path.join(dataset_dir, 't10k-images-idx3-ubyte')
y_test_path = os.path.join(dataset_dir, 't10k-labels-idx1-ubyte')

with urllib.request.urlopen(X_train_url) as response:
    with open(X_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_train_url) as response:
    with open(y_train_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(X_test_url) as response:
    with open(X_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

with urllib.request.urlopen(y_test_url) as response:
    with open(y_test_path, 'wb') as out_file:
        data_gzip = response.read()
        data = gzip.decompress(data_gzip)
        out_file.write(data)

In [10]:
import mindspore.dataset as ds

train_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='train', shuffle=True)
test_ds = ds.FashionMnistDataset(dataset_dir=dataset_dir, usage='test', shuffle=True)

In [11]:
import mindspore.dataset.vision as vision
import mindspore.dataset.transforms as transforms
from mindspore import dtype as mstype

def transform_ds(dataset):
    image_transforms = [
        vision.Resize(size=(224, 224)),
        vision.Rescale(rescale=1/255, shift=0),
        vision.HWC2CHW()
    ]
    label_transforms = [
        transforms.OneHot(num_classes=10),
        transforms.TypeCast(data_type=mstype.float32)
    ]
    dataset = dataset.map(operations=image_transforms, input_columns='image')
    dataset = dataset.map(operations=label_transforms, input_columns='label')
    dataset = dataset.batch(batch_size=128, drop_remainder=False)
    return dataset

train_ds = transform_ds(dataset=train_ds)
test_ds = transform_ds(dataset=test_ds)

In [12]:
loss_fn = nn.SoftmaxCrossEntropyWithLogits(reduction='mean')
loss_fn

SoftmaxCrossEntropyWithLogits()

In [13]:
optimizer = nn.SGD(params=nin_full_amp.trainable_params(), learning_rate=0.01)
optimizer

SGD()

Here, let's apply a loss scale manager with [`mindspore.amp.FixedLossScaleManager`](https://www.mindspore.cn/docs/en/r2.9.0/api_python/amp/mindspore.amp.FixedLossScaleManager.html#mindspore.amp.FixedLossScaleManager), specifying a fixed loss scale of `1024.0`. This together with batch normalization should help prevent gradient underflow and overflow with MindSpore's `O2` mixed precision.

In [14]:
loss_scale_manager = amp.FixedLossScaleManager(loss_scale=1024.0)
loss_scale_manager

In [15]:
from mindspore.train import Model

model = Model(network=nin_full_amp,
              loss_fn=loss_fn,
              optimizer=optimizer,
              metrics={'accuracy', 'loss'},
              loss_scale_manager=loss_scale_manager)
model

In [16]:
from mindspore.train import EarlyStopping

early_stopping = EarlyStopping(patience=5, verbose=True, restore_best_weights=True)
early_stopping

In [17]:
epochs = 100

In [18]:
model.fit(epoch=epochs,
          train_dataset=train_ds,
          valid_dataset=test_ds,
          callbacks=[early_stopping],
          dataset_sink_mode=True)

path string is NULLpath string is NULL...Restoring model weights from the end of the best epoch.
Epoch 00011: early stopping


Let's check the validation loss and accuracy of our trained NiN model against the Fashion MNIST dataset.

In [19]:
metrics = model.eval(valid_dataset=test_ds)
val_acc = metrics['accuracy']
val_loss = metrics['loss']
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_acc:.4f}')

Validation loss: 0.3058
Validation accuracy: 0.9015


Our trained NiN model easily reaches $90\%$ accuracy as well similar to VGG-11. Excellent!

## 8.3.4. Summary

We saw in this chapter the NiN architecture and how it introduced 2 revolutionary ideas at the time that influenced the development of modern CNNs.

1. $1 \times 1$ convolutions as per-pixel FC layers acting upon the channel dimension for local nonlinearities
1. Applying a global average pooling operation followed by flattening to obtain the final logits, obviating the need for massive FC layers at the head of the neural network